# August 2026 verification · **05 — Gold: is the tail really diverging from the list?**

**The claim under test:** the total waiting list grew by well under one per
cent while the over-52-week tail grew by roughly six, "like separate queues".

The gate (04) already showed that July to August is +0.57% for the list and
+1.8% for the tail on NHS England's headline basis. So this notebook asks
the longer question properly: over **which window**, on **which measure**, is
the tail outgrowing the list?

**Two windows**
* **April 2025 = 100**: the window the article brief proposed.
* **March 2026 = 100**: the trough of the over-52-week series, and the date of two national targets.

**Three bases**
* **reported**: my primary measure, the Two Maps measure (reported providers, NONC excluded).
* **constant panel**: only providers that reported in *every* month of the window. This kills the submitter artefact: Sheffield, Torbay, Plymouth, Northampton and Cora dropping out, and some 50 independent-sector providers joining.
* **headline**: NHS England's figure including carry-forward estimates for missing trusts, as rebuilt and proven exact in the gate.

Then I build the three-row table for August 2026, and finally investigate
March 2026: was the trough a revision artefact, a few providers, or something
system-wide?

Outputs go to `data/gold/05_divergence/`.

In [1]:
import sys
import zipfile
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import (BRONZE_DIR, GOLD_DIR, RTT_UNIDENTIFIED_COMMISSIONER,  # noqa: E402
                     SILVER_DIR, TRUST_SUCCESSIONS)

OUT = GOLD_DIR / "05_divergence"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

MEASURES = ["total", "over_18", "over_52"]
WINDOWS = {"Apr25": "2025-04", "Mar26": "2026-03"}
LAST = "2026-08"

## Step 0 — The gate must have passed

I don't take that on trust. I read the gate's own output and refuse to run
if a single check failed.

In [2]:
gate = pd.read_csv(GOLD_DIR / "00_reconciliation" / "reconciliation.csv")
assert gate["pass"].all(), f"reconciliation gate has {(~gate['pass']).sum()} failing checks: run 04 first"
print(f"gate: {len(gate)} of {len(gate)} checks passed")

inc = pd.read_parquet(SILVER_DIR / "rtt" / "rtt_provider.parquet").query("part == 'incomplete'")
pbands = pd.read_parquet(SILVER_DIR / "rtt" / "rtt_provider_bands.parquet")
headline = pd.read_csv(GOLD_DIR / "00_reconciliation" / "headline_rebuilt.csv", index_col="period")
periods = sorted(inc.period.unique())

gate: 51 of 51 checks passed


## Step 1 — The three bases, month by month

**Constant panel.** For the panel I need entities that exist throughout. North
Bristol (RVJ) merged into Bristol NHS FT (RA7) in July 2026, so for panel
purposes I add RVJ to RA7 in **every** month, not just from July. The merged
trust is then one continuous entity, rather than RVJ vanishing and RA7
doubling. A provider is "present" in a month only if it reported at least one
incomplete pathway.

In [3]:
def panel_entity(codes: pd.Series) -> pd.Series:
    # Undated on purpose: the panel needs one continuous entity per merger.
    return codes.replace({old: new for old, new, _ in TRUST_SUCCESSIONS})

inc = inc.assign(entity=panel_entity(inc.provider_code))
ent = inc.groupby(["period", "entity"])[MEASURES].sum().reset_index()
present = ent[ent.total > 0].pivot_table(index="entity", columns="period", values="total", aggfunc="size").notna()

def panel_members(start: str) -> pd.Index:
    cols = [p for p in periods if p >= start]
    return present.index[present[cols].all(axis=1)]

panels = {w: panel_members(start) for w, start in WINDOWS.items()}

series = {}
series["reported"] = inc.groupby("period")[MEASURES].sum()
for w, members in panels.items():
    series[f"panel_{w}"] = ent[ent.entity.isin(members)].groupby("period")[MEASURES].sum()
series["headline"] = headline.rename(columns={"total_est": "total_h", "over_18_est": "over_18_h", "over_52_est": "over_52_h"})[
    ["total_h", "over_18_h", "over_52_h"]].set_axis(MEASURES, axis=1)

# How much of the list does each panel hold? A panel that keeps 60% of the
# list would be answering a different question.
cov = []
for w, members in panels.items():
    s = series[f"panel_{w}"]
    for p in [WINDOWS[w], LAST]:
        cov.append({"panel": w, "period": p, "providers": len(members),
                    "share_of_reported_total": s.loc[p, "total"] / series["reported"].loc[p, "total"],
                    "share_of_reported_over_52": s.loc[p, "over_52"] / series["reported"].loc[p, "over_52"]})
coverage = pd.DataFrame(cov)
coverage

,panel,period,providers,share_of_reported_total,share_of_reported_over_52
0,Apr25,2025-04,448,0.96,0.97
1,Apr25,2026-08,448,0.98,0.99
2,Mar26,2026-03,520,0.98,0.98
3,Mar26,2026-08,520,1.00,1.00


In [4]:
# Who is outside each panel, and how big are they? Largest first.
last_seen = inc.groupby("entity").agg(name=("provider_name", "last"), max_total=("total", "max"))
for w, members in panels.items():
    out = last_seen[~last_seen.index.isin(members)].sort_values("max_total", ascending=False)
    print(f"panel {w}: {len(out)} entities excluded; the largest:")
    print(out.head(8).to_string(), "\n")

panel Apr25: 103 entities excluded; the largest:
                                                                  name  max_total
entity                                                                           
RHQ                  SHEFFIELD TEACHING HOSPITALS NHS FOUNDATION TRUST  92,037.00
RF4     BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOSPITALS NHS TRUST  72,238.00
RK9                            UNIVERSITY HOSPITALS PLYMOUTH NHS TRUST  49,961.00
RNS                             NORTHAMPTON GENERAL HOSPITAL NHS TRUST  44,903.00
RA9                        TORBAY AND SOUTH DEVON NHS FOUNDATION TRUST  34,622.00
NMG                      CORA HEALTH MSK (FORMERLY CONNECT HEALTH LTD)  24,720.00
NNH                                                         OPTEGRA UK   7,983.00
S5E5Y                       HCRG CARE DERMATOLOGY THE IRONSTONE CENTRE   4,534.00 

panel Mar26: 31 entities excluded; the largest:
                                                     name  max_total
entity      

## Step 2 — Indices for both windows, all three bases

In [5]:
rows = []
for w, start in WINDOWS.items():
    for basis in ["reported", f"panel_{w}", "headline"]:
        s = series[basis]
        s = s[s.index >= start]
        idx = 100 * s / s.loc[start]
        for p, r in idx.iterrows():
            rows.append({"window": w, "basis": basis.replace(f"_{w}", ""), "period": p,
                         **{f"{m}_index": r[m] for m in MEASURES}, **{m: s.loc[p, m] for m in MEASURES}})
index_long = pd.DataFrame(rows)
index_long.to_csv(OUT / "index_long.csv", index=False)
index_long.pivot_table(index="period", columns=["window", "basis"], values=["total_index", "over_52_index"]).round(1)

over_52_index                                          total_index                                         
window          Apr25                    Mar26                       Apr25                    Mar26                
basis        headline  panel reported headline  panel reported    headline  panel reported headline  panel reported
period                                                                                                             
2025-04        100.00 100.00   100.00      NaN    NaN      NaN      100.00 100.00   100.00      NaN    NaN      NaN
2025-05        103.60 103.90   103.60      NaN    NaN      NaN       99.60  99.60    99.60      NaN    NaN      NaN
2025-06        100.90 101.10   100.90      NaN    NaN      NaN       99.70  99.60    99.70      NaN    NaN      NaN
2025-07        100.60 100.70    99.70      NaN    NaN      NaN      100.00  99.90    98.80      NaN    NaN      NaN
2025-08        100.30 100.50    99.30      NaN    NaN      NaN      100.20 100.10    99.00      NaN    NaN      NaN
2025-09         94.90  94.90    94.00      NaN    NaN      NaN      100.00  99.90    98.80      NaN    NaN      NaN
2025-10         90.80  90.60    89.90      NaN    NaN      NaN      100.10 100.00    98.90      NaN    NaN      NaN
2025-11         82.30  82.30    81.10      NaN    NaN      NaN       98.90  98.70    96.90      NaN    NaN      NaN
2025-12         73.90  73.70    73.00      NaN    NaN      NaN       98.70  98.30    97.40      NaN    NaN      NaN
2026-01         71.40  71.10    70.40      NaN    NaN      NaN       98.10  97.70    96.80      NaN    NaN      NaN
2026-02         64.60  64.10    63.60      NaN    NaN      NaN       97.70  97.30    96.40      NaN    NaN      NaN
2026-03         49.70  49.00    48.70   100.00 100.00   100.00       96.20  95.80    94.90   100.00 100.00   100.00
2026-04         52.50  51.90    51.30   105.70 106.00   105.20       97.70  97.20    96.00   101.60 101.60   101.10
2026-05         55.10  54.60    53.90   110.90 111.40   110.50       98.50  98.10    96.80   102.40 102.40   102.00
2026-06         55.60  55.10    54.40   112.00 112.50   111.60       98.40  97.90    96.70   102.30 102.30   101.90
2026-07         58.60  58.10    57.00   118.00 118.80   116.90       99.20  98.80    96.50   103.10 103.10   101.70
2026-08         59.60  59.10    57.90   120.10 120.90   118.70       99.70  99.40    96.90   103.70 103.70   102.00

## Step 3 — The divergence, stated as numbers

For each window and basis: the % change from the base month to August 2026
for the list and for the tail, the gap between them in percentage points, and
the number of months in which the tail grew faster than the list, month on month.

In [6]:
summ = []
for (w, basis), g in index_long.groupby(["window", "basis"], sort=False):
    g = g.set_index("period").sort_index()
    mom = g[MEASURES].pct_change().dropna() * 100
    summ.append({
        "window": w, "basis": basis,
        "total_change_pct": g.loc[LAST, "total_index"] - 100,
        "over_18_change_pct": g.loc[LAST, "over_18_index"] - 100,
        "over_52_change_pct": g.loc[LAST, "over_52_index"] - 100,
        "gap_pp": (g.loc[LAST, "over_52_index"] - 100) - (g.loc[LAST, "total_index"] - 100),
        "months": len(mom),
        "months_tail_faster": int((mom.over_52 > mom.total).sum()),
        "months_tail_rose": int((mom.over_52 > 0).sum()),
    })
divergence = pd.DataFrame(summ)
divergence.to_csv(OUT / "divergence_summary.csv", index=False)
divergence

,window,basis,total_change_pct,over_18_change_pct,over_52_change_pct,gap_pp,months,months_tail_faster,months_tail_rose
0,Apr25,reported,-3.12,-16.29,-42.15,-39.02,16,6,6
1,Apr25,panel,-0.65,-14.30,-40.91,-40.26,16,6,6
2,Apr25,headline,-0.26,-13.83,-40.36,-40.10,16,6,6
3,Mar26,reported,2.04,2.30,18.71,16.67,5,5,5
4,Mar26,panel,3.74,4.05,20.86,17.12,5,5,5
5,Mar26,headline,3.72,4.00,20.06,16.34,5,5,5


### Month on month, side by side

This is the form the claim takes ("the list grew under 1%, the tail
roughly 6%"), so I show it every month, on every basis.

In [7]:
mom_rows = []
for basis in ["reported", "panel_Apr25", "headline"]:
    s = series[basis]
    pct = s.pct_change() * 100
    for p in periods[1:]:
        mom_rows.append({"period": p, "basis": basis, "total_mom_pct": pct.loc[p, "total"],
                         "over_52_mom_pct": pct.loc[p, "over_52"],
                         "over_52_mom_abs": s.loc[p, "over_52"] - s.loc[s.index[s.index.get_loc(p) - 1], "over_52"]})
mom = pd.DataFrame(mom_rows)
mom.to_csv(OUT / "month_on_month.csv", index=False)
mom.pivot_table(index="period", columns="basis", values=["total_mom_pct", "over_52_mom_pct"]).round(2)

over_52_mom_pct                      total_mom_pct                     
basis          headline panel_Apr25 reported      headline panel_Apr25 reported
period                                                                         
2025-05            3.59        3.87     3.59         -0.45       -0.44    -0.45
2025-06           -2.58       -2.64    -2.58          0.16        0.03     0.16
2025-07           -0.28       -0.45    -1.22          0.30        0.26    -0.93
2025-08           -0.37       -0.21    -0.38          0.18        0.24     0.18
2025-09           -5.34       -5.51    -5.39         -0.21       -0.22    -0.21
2025-10           -4.31       -4.54    -4.35          0.10        0.11     0.10
2025-11           -9.31       -9.22    -9.74         -1.17       -1.25    -1.99
2025-12          -10.21      -10.38   -10.00         -0.25       -0.40     0.57
2026-01           -3.45       -3.54    -3.50         -0.60       -0.61    -0.61
2026-02           -9.57       -9.80    -9.70         -0.43       -0.46    -0.43
2026-03          -23.04      -23.60   -23.38         -1.53       -1.56    -1.55
2026-04            5.70        5.93     5.16          1.58        1.54     1.12
2026-05            4.96        5.18     5.09          0.83        0.85     0.85
2026-06            0.93        0.84     0.95         -0.08       -0.13    -0.09
2026-07            5.36        5.47     4.79          0.77        0.82    -0.22
2026-08            1.76        1.77     1.55          0.57        0.61     0.37

In [8]:
# The figures the article brief used, located: is there ANY single month, or
# any two-month span, in which the tail grew ~6% or ~7,000 on the headline
# basis while the list grew under 1%?
h = series["headline"]
spans = []
for k in (1, 2, 3):
    d = h - h.shift(k)
    pc = (h / h.shift(k) - 1) * 100
    for p in h.index[k:]:
        spans.append({"months": k, "from": h.index[h.index.get_loc(p) - k], "to": p,
                      "over_52_abs": d.loc[p, "over_52"], "over_52_pct": pc.loc[p, "over_52"], "total_pct": pc.loc[p, "total"]})
spans = pd.DataFrame(spans)
near = spans[(spans.over_52_pct.between(5, 7.5)) | (spans.over_52_abs.between(6_000, 8_000))]
print("headline-basis spans where the tail moved ~6% or ~7,000:")
near.round(2)

headline-basis spans where the tail moved ~6% or ~7,000:


,months,from,to,over_52_abs,over_52_pct,total_pct
0,1,2025-04,2025-05,"6,816.00",3.59,-0.45
11,1,2026-03,2026-04,"5,377.00",5.70,1.58
14,1,2026-06,2026-07,"5,668.00",5.36,0.77
28,2,2026-04,2026-06,"5,930.00",5.94,0.75
29,2,2026-05,2026-07,"6,645.00",6.34,0.69
30,2,2026-06,2026-08,"7,628.00",7.22,1.34


## Step 4 — The three-row table, August 2026

Total list, over 18 weeks, over 52 weeks. Since the March 2026 target deadline,
month on month (vs July 2026) and year on year (vs August 2025), absolute
and %, so a reader can see the
denominator effect for themselves. Year on year on the Mar26 panel isn't
possible (it starts in March), so the panel row uses the Apr25 panel.

In [9]:
tbl = []
labels = {"total": "Total list", "over_18": "Over 18 weeks", "over_52": "Over 52 weeks"}
for basis in ["reported", "panel_Apr25", "headline"]:
    s = series[basis]
    for m in MEASURES:
        aug, jul, aug25, mar = s.loc[LAST, m], s.loc["2026-07", m], s.loc["2025-08", m], s.loc["2026-03", m]
        tbl.append({"basis": basis, "measure": labels[m], "aug_2026": aug,
                    "since_mar_abs": aug - mar, "since_mar_pct": 100 * (aug / mar - 1),
                    "mom_abs": aug - jul, "mom_pct": 100 * (aug / jul - 1),
                    "yoy_abs": aug - aug25, "yoy_pct": 100 * (aug / aug25 - 1)})
three_row = pd.DataFrame(tbl)
three_row.to_csv(OUT / "three_row_table.csv", index=False)
three_row.round(2)

,basis,measure,aug_2026,since_mar_abs,since_mar_pct,mom_abs,mom_pct,yoy_abs,yoy_pct
0,reported,Total list,"7,158,164.00","143,329.00",2.04,"26,479.00",0.37,"-154,192.00",-2.11
1,reported,Over 18 weeks,"2,490,610.00","56,095.00",2.30,"15,984.00",0.65,"-363,292.00",-12.73
2,reported,Over 52 weeks,"109,936.00","17,329.00",18.71,"1,673.00",1.55,"-78,766.00",-41.74
3,panel_Apr25,Total list,"7,042,997.00","253,878.00",3.74,"42,646.00",0.61,"-52,798.00",-0.74
4,panel_Apr25,Over 18 weeks,"2,461,619.00","97,024.00",4.10,"27,754.00",1.14,"-325,838.00",-11.69
5,panel_Apr25,Over 52 weeks,"108,555.00","18,538.00",20.59,"1,890.00",1.77,"-75,989.00",-41.18
6,headline,Total list,"7,370,100.00","264,009.00",3.72,"41,848.00",0.57,"-33,512.00",-0.45
7,headline,Over 18 weeks,"2,564,009.00","98,501.00",4.00,"28,324.00",1.12,"-320,886.00",-11.12
8,headline,Over 52 weeks,"113,339.00","18,935.00",20.06,"1,960.00",1.76,"-77,160.00",-40.50


## Step 5 — March 2026: what happened?

The over-52-week series falls hard into March 2026 and rises every month
after it. March is also one of the two months NHS England revised on 9 July
2026. Three explanations to separate:

1. **a revision artefact**: the revised March differs from the original;
2. **a few providers**: one or two trusts' data changed;
3. **something system-wide at year end**, such as a validation push or a treatment push towards a target.

### 5a — Did the July revision change March? Original vintage vs revised

I read the original vintages (still hosted on england.nhs.uk, found through
the Wayback Machine's copy of the 2025-26 page) through exactly the Silver
measure.

In [10]:
def measure_extract(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as z:
        name = [n for n in z.namelist() if n.endswith(".csv")][0]
        with z.open(name) as f:
            d = pd.read_csv(f, low_memory=False)
    d = d[(d["RTT Part Description"] == "Incomplete Pathways") & (d["Treatment Function Code"] == "C_999")
          & ~d["Commissioner Org Code"].astype(str).str.startswith(RTT_UNIDENTIFIED_COMMISSIONER)]
    b = [c for c in d.columns if c.startswith("Gt ")]
    d = d.assign(total=d[b].sum(axis=1), within_18=d[b[:18]].sum(axis=1),
                 over_52=d[[c for c in b if int(c.split()[1]) >= 52]].sum(axis=1))
    return d.groupby("Provider Org Code")[["total", "within_18", "over_52"]].sum()

rev = []
for p in ["2025-12", "2026-03"]:
    orig = measure_extract(next((BRONZE_DIR / f"nhse_rtt_superseded_{p}").glob("*.zip")))
    revd = measure_extract(next((BRONZE_DIR / f"nhse_rtt_{p}").glob("*.zip")))
    diff = revd.sub(orig, fill_value=0)
    changed = diff[(diff != 0).any(axis=1)]
    rev.append({"period": p, "orig_total": orig.total.sum(), "revised_total": revd.total.sum(),
                "orig_over_52": orig.over_52.sum(), "revised_over_52": revd.over_52.sum(),
                "providers_changed": len(changed), "which": ", ".join(changed.index)})
revision = pd.DataFrame(rev)
revision.to_csv(OUT / "march_revision_effect.csv", index=False)
revision

,period,orig_total,revised_total,orig_over_52,revised_over_52,providers_changed,which
0,2025-12,"7,199,624.00","7,199,624.00","138,711.00","138,711.00",0,
1,2026-03,"7,014,879.00","7,014,835.00","92,609.00","92,607.00",1,RAJ


### 5b — Was the March fall broad or concentrated?

Provider-level change in the over-52 count, month by month, on the Apr25
panel: how many providers' tails fell, and what share of the national fall
the ten largest fallers account for.

In [11]:
pe = ent[ent.entity.isin(panels["Apr25"])].pivot_table(index="entity", columns="period", values="over_52", aggfunc="sum")
breadth = []
for a, b in zip(periods[:-1], periods[1:]):
    d = pe[b] - pe[a]
    fall = d[d < 0]
    breadth.append({"to": b, "net_change": d.sum(), "providers_falling": int((d < 0).sum()),
                    "providers_rising": int((d > 0).sum()),
                    "top10_share_of_falls": fall.nsmallest(10).sum() / fall.sum() if len(fall) else np.nan})
breadth = pd.DataFrame(breadth)
breadth.to_csv(OUT / "march_breadth.csv", index=False)
breadth.round(3)

,to,net_change,providers_falling,providers_rising,top10_share_of_falls
0,2025-05,"7,104.00",129,146,0.52
1,2025-06,"-5,029.00",132,126,0.45
2,2025-07,-842.00,126,135,0.52
3,2025-08,-397.00,135,134,0.50
4,2025-09,"-10,170.00",155,108,0.42
5,2025-10,"-7,910.00",170,97,0.37
6,2025-11,"-15,349.00",183,84,0.36
7,2025-12,"-15,680.00",177,97,0.39
8,2026-01,"-4,799.00",138,139,0.43
9,2026-02,"-12,805.00",178,99,0.42


### 5c — The tail cohort: how many long waiters left the list each month?

The over-52 figure is a stock. To see flow, I follow the cohort. A pathway
waiting at least 48 weeks at the end of month *t−1* is certain to be over 52
weeks at the end of month *t* if it is still waiting (every month is at least
28 days, i.e. at least 4 weeks). So on the panel:

> **exits ≥ over-52 at *t−1* + those in the 48-52-week bands at *t−1* − over-52 at *t***

That is a lower bound on long waiters leaving the list, by treatment *or*
validation removal; the data can't tell those apart. I also express it as a
share of the cohort at risk.

In [12]:
wcols = [f"w{k:03d}" for k in range(105)]
pb = pbands.assign(entity=panel_entity(pbands.provider_code))
pb = pb[pb.entity.isin(panels["Apr25"])].groupby("period")[wcols].sum()
o52 = pb[[f"w{k:03d}" for k in range(52, 105)]].sum(axis=1)
near52 = pb[[f"w{k:03d}" for k in range(48, 52)]].sum(axis=1)
# consistency: the band table on the panel must equal the panel's over-52 series
assert (o52 == series["panel_Apr25"].over_52).all()
cohort = pd.DataFrame({"over_52_prev": o52.shift(1), "crossing_in": near52.shift(1), "over_52_now": o52})
cohort["exits_lower_bound"] = cohort.over_52_prev + cohort.crossing_in - cohort.over_52_now
cohort["exit_share_of_cohort"] = cohort.exits_lower_bound / (cohort.over_52_prev + cohort.crossing_in)
cohort = cohort.dropna()
cohort.to_csv(OUT / "tail_cohort_exits.csv")
cohort.round(3)

,over_52_prev,crossing_in,over_52_now,exits_lower_bound,exit_share_of_cohort
period,,,,,
2025-05,"183,708.00","121,942.00","190,812.00","114,838.00",0.38
2025-06,"190,812.00","132,769.00","185,783.00","137,798.00",0.43
2025-07,"185,783.00","125,745.00","184,941.00","126,587.00",0.41
2025-08,"184,941.00","110,900.00","184,544.00","111,297.00",0.38
2025-09,"184,544.00","124,128.00","174,374.00","134,298.00",0.43
2025-10,"174,374.00","123,246.00","166,464.00","131,156.00",0.44
2025-11,"166,464.00","119,260.00","151,115.00","134,609.00",0.47
2025-12,"151,115.00","92,822.00","135,435.00","108,502.00",0.45
2026-01,"135,435.00","105,133.00","130,636.00","109,932.00",0.46


### 5d — List-wide flows from NHS England's own series

The Overview series publishes completed pathways, clock starts and **derived
unreported removals**: the balancing item, i.e. pathways that left the list
without a reported completion (validation, among other things). NHS
England's definition: start list + clock starts − completions − end list.
Headers repeat across groups ("No. of pathways (all)" appears under admitted
and non-admitted), so I key each column on its group header plus its own.

In [13]:
ws = openpyxl.load_workbook(BRONZE_DIR / "nhse_rtt_published_2026-08" /
                            "RTT-Overview-Timeseries-Including-Estimates-for-Missing-Trusts-Aug26-XLS-139K-vjgRdG.xlsx",
                            read_only=True, data_only=True).worksheets[0]
orows = list(ws.iter_rows(values_only=True))
group, keys = None, []
for g_, h_ in zip(orows[10], orows[11]):
    group = str(g_).strip() if g_ else group
    keys.append(f"{group} | {str(h_).strip()}" if h_ else None)
want = {
    "completed_admitted": "Completed admitted (unadjusted) RTT pathways | No. of pathways (all) with estimates for missing data",
    "completed_nonadmitted": "Completed non-admitted RTT pathways | No. of pathways (all) with estimates for missing data",
    "clock_starts": "New RTT periods | No. of new RTT periods with estimates for missing data",
    "unreported_removals": "Derived unreported completed pathways and removals | Total unreported completed pathways and removals with estimates",
}
pos = {k: keys.index(v) for k, v in want.items()}
flows = pd.DataFrame([{**{k: r[i] for k, i in pos.items()}, "period": r[2].strftime("%Y-%m")}
                      for r in orows[12:] if hasattr(r[2], "strftime")]).set_index("period").loc[periods].astype(float)
flows["completed"] = flows.completed_admitted + flows.completed_nonadmitted
flows["removals_share"] = flows.unreported_removals / (flows.completed + flows.unreported_removals)
# each flow as a multiple of its own median over the window, so March is easy to read
rel = (flows[["completed", "clock_starts", "unreported_removals"]] / flows[["completed", "clock_starts", "unreported_removals"]].median()).add_suffix("_vs_median")
flows = flows.join(rel)
flows.to_csv(OUT / "march_flows.csv")
flows[["completed", "clock_starts", "unreported_removals", "removals_share",
       "completed_vs_median", "unreported_removals_vs_median"]].round(3)

,completed,clock_starts,unreported_removals,removals_share,completed_vs_median,unreported_removals_vs_median
period,,,,,,
2025-04,"1,450,094.00","1,707,444.00","286,883.00",0.17,0.94,1.10
2025-05,"1,500,695.00","1,757,217.00","289,637.00",0.16,0.98,1.11
2025-06,"1,558,632.00","1,828,309.00","257,791.00",0.14,1.01,0.99
2025-07,"1,648,944.52","1,942,537.33","271,149.81",0.14,1.07,1.04
2025-08,"1,404,088.24","1,626,863.33","209,442.10",0.13,0.91,0.81
2025-09,"1,617,382.76","1,845,641.67","243,721.90",0.13,1.05,0.94
2025-10,"1,677,868.52","1,930,201.33","245,036.81",0.13,1.09,0.94
2025-11,"1,536,543.11","1,710,325.42","260,299.31",0.14,1.00,1.00
2025-12,"1,426,850.00","1,647,000.00","238,198.00",0.14,0.93,0.92


### 5e — The two national targets dated March 2026

From NHS England's *2025/26 priorities and operational planning guidance*
(england.nhs.uk/long-read/2025-26-priorities-and-operational-planning-guidance/,
read 16 October 2026): 18-week performance to **65% nationally by March 2026**,
and the proportion waiting over 52 weeks to **less than 1% of the total waiting
list by March 2026**. Both are measured here on the headline basis, the basis
NHS England reports performance on.

In [14]:
TARGETS = {"pct_within_18": 65.0, "pct_over_52": 1.0}
hb = headline.loc[["2026-02", "2026-03", "2026-04", LAST]]
targets = pd.DataFrame({
    "pct_within_18": 100 * (hb.total_est - hb.over_18_est) / hb.total_est,
    "pct_over_52": 100 * hb.over_52_est / hb.total_est,
})
targets["meets_18wk_target"] = targets.pct_within_18 >= TARGETS["pct_within_18"]
targets["meets_52wk_target"] = targets.pct_over_52 < TARGETS["pct_over_52"]
targets.to_csv(OUT / "march_targets.csv")
targets.round(2)

,pct_within_18,pct_over_52,meets_18wk_target,meets_52wk_target
period,,,,
2026-02,62.59,1.70,False,False
2026-03,65.30,1.33,True,False
2026-04,64.97,1.38,False,False
2026-08,65.21,1.54,True,False


## Step 6 — Verdict, computed

Every sentence below is printed from the tables above. None is typed in.

In [15]:
dv = divergence.set_index(["window", "basis"])
for w in WINDOWS:
    for basis in ["reported", "panel", "headline"]:
        r = dv.loc[(w, basis)]
        direction = "FASTER than" if r.gap_pp > 0 else "SLOWER than (or opposite to)"
        print(f"[{w}=100, {basis:8s}] list {r.total_change_pct:+6.1f}% | over-52 {r.over_52_change_pct:+6.1f}% | "
              f"tail grew {direction} the list by {abs(r.gap_pp):.1f} pp | tail faster in {int(r.months_tail_faster)}/{int(r.months)} months")
mr = revision.set_index("period").loc["2026-03"]
print(f"\nMarch 2026 revision: over-52 {mr.orig_over_52:,.0f} -> {mr.revised_over_52:,.0f}, total "
      f"{mr.orig_total:,.0f} -> {mr.revised_total:,.0f} ({mr.providers_changed} provider changed: {mr.which})")
b3 = breadth.set_index("to").loc["2026-03"]
print(f"March fall breadth (Apr25 panel): {int(b3.providers_falling)} providers fell, {int(b3.providers_rising)} rose; "
      f"ten largest fallers = {b3.top10_share_of_falls:.0%} of the falls")
c3 = cohort.loc["2026-03"]; cmed = cohort.exit_share_of_cohort.drop("2026-03").median()
print(f"tail cohort exits, March: at least {c3.exits_lower_bound:,.0f} ({c3.exit_share_of_cohort:.1%} of the at-risk cohort; "
      f"median of the other months {cmed:.1%})")
f3 = flows.loc["2026-03"]
print(f"list-wide in March: completions {f3.completed_vs_median:.2f}x the window median, unreported removals "
      f"{f3.unreported_removals_vs_median:.2f}x; removals were {f3.removals_share:.1%} of all exits")
t3 = targets.loc["2026-03"]
print(f"March 2026 targets (headline): {t3.pct_within_18:.1f}% within 18 weeks (target 65%: "
      f"{'met' if t3.meets_18wk_target else 'missed'}); {t3.pct_over_52:.2f}% over 52 weeks (target <1%: "
      f"{'met' if t3.meets_52wk_target else 'missed'})")

[Apr25=100, reported] list   -3.1% | over-52  -42.1% | tail grew SLOWER than (or opposite to) the list by 39.0 pp | tail faster in 6/16 months
[Apr25=100, panel   ] list   -0.6% | over-52  -40.9% | tail grew SLOWER than (or opposite to) the list by 40.3 pp | tail faster in 6/16 months
[Apr25=100, headline] list   -0.3% | over-52  -40.4% | tail grew SLOWER than (or opposite to) the list by 40.1 pp | tail faster in 6/16 months
[Mar26=100, reported] list   +2.0% | over-52  +18.7% | tail grew FASTER than the list by 16.7 pp | tail faster in 5/5 months
[Mar26=100, panel   ] list   +3.7% | over-52  +20.9% | tail grew FASTER than the list by 17.1 pp | tail faster in 5/5 months
[Mar26=100, headline] list   +3.7% | over-52  +20.1% | tail grew FASTER than the list by 16.3 pp | tail faster in 5/5 months

March 2026 revision: over-52 92,609 -> 92,607, total 7,014,879 -> 7,014,835 (1 provider changed: RAJ)
March fall breadth (Apr25 panel): 202 providers fell, 73 rose; ten largest fallers = 30% of t